# Module 5b: Montages & Dipoles -- Why the Same Brain Looks Different on Different Screens

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup & Imports

This notebook covers **EEG montages** -- the mathematical display method that transforms raw electrode recordings into the traces you see on a screen.

Key concepts:
- **Referential montage**: each electrode compared to one common reference
- **Bipolar montage**: neighboring electrodes subtracted from each other
- **Phase reversal**: the key pattern for localizing a spike source
- **The double banana**: the standard clinical bipolar montage

Remember: EEG always measures voltage *differences*. $$V_{\text{channel}} = V_{\text{electrode A}} - V_{\text{electrode B}}$$
The montage defines which electrode is A and which is B.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
%matplotlib inline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

**Exercise 1:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 2: Simulating 5 Electrodes with a Focal Source

We simulate a chain of 5 electrodes along the left parasagittal chain: **Fp1, F3, C3, P3, O1** (front to back).

A focal electrical source (a spike) is placed at **C3**. The spike is modeled as a sharp negative transient that is strongest at the source electrode and falls off with distance. This simulates a cortical dipole generating a field that nearby electrodes detect.

In [ ]:
# Simulate electrode voltages with a focal spike at C3
np.random.seed(42)
fs = 500  # Sampling rate
duration = 2  # seconds
t = np.arange(0, duration, 1/fs)
N = len(t)

electrodes = ['Fp1', 'F3', 'C3', 'P3', 'O1']
n_elec = len(electrodes)

def generate_electrode_voltages(source_idx, spike_time=1.0):
    """
    Generate voltage at each electrode.
    source_idx: index of the electrode closest to the spike source.
    """
    voltages = {}
    for i, name in enumerate(electrodes):
        # Background: alpha + theta + noise
        bg = (15 * np.sin(2*np.pi*10*t + i*0.7) +
              8 * np.sin(2*np.pi*6*t + i*1.3) +
              5 * np.random.randn(N))
        
        # Spike: Gaussian-windowed sharp transient at the source
        dist = abs(i - source_idx)
        spike = np.zeros(N)
        if dist <= 2:
            falloff = [1.0, 0.4, 0.1][dist]
            t_rel = t - spike_time
            spike_width = 0.03  # 30ms
            mask = np.abs(t_rel) < 0.15
            spike[mask] = (-120 * falloff * 
                          np.exp(-0.5 * (t_rel[mask]/spike_width)**2))
            # Slow after-wave
            spike[mask] += (40 * falloff * 
                           np.exp(-0.5 * ((t_rel[mask]-0.06)/0.05)**2))
        voltages[name] = bg + spike
    return voltages

# Generate with source at C3 (index 2)
source_idx = 2
voltages = generate_electrode_voltages(source_idx)

# Plot raw electrode voltages
fig, ax = plt.subplots(figsize=(12, 6))
offset = 0
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for i, name in enumerate(electrodes):
    ax.plot(t, voltages[name] + offset, color=colors[i], linewidth=1.2)
    ax.text(-0.05, offset, name, fontsize=11, fontweight='bold',
            color=colors[i], ha='right', va='center')
    offset -= 120

ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='Spike at t=1s')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_title(f'Raw Electrode Voltages (Source at {electrodes[source_idx]})',
             fontsize=14, fontweight='bold')
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.show()
print(f'Spike source is at electrode {electrodes[source_idx]}.')
print('Notice: C3 has the largest deflection, neighboring electrodes show smaller versions.')

**Exercise 2:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 3: Referential Montage (Each Electrode vs Reference)

In a **referential montage**, every electrode is compared against one common reference (often a linked-ears average, Cz, or a computed average reference).

$$V_{\text{channel}} = V_{\text{electrode}} - V_{\text{reference}}$$

**Advantage**: Easy to see absolute voltage at each electrode. The electrode with the **biggest deflection** is closest to the source.

**Disadvantage**: If the reference electrode is contaminated (e.g., by artifact), all channels are affected.

In [ ]:
# Referential montage: each electrode minus a common reference
# We use the average of all electrodes as reference (average reference)
avg_ref = np.mean([voltages[e] for e in electrodes], axis=0)

ref_montage = {}
for name in electrodes:
    ref_montage[name] = voltages[name] - avg_ref

# Plot referential montage
fig, ax = plt.subplots(figsize=(12, 6))
offset = 0
for i, name in enumerate(electrodes):
    ax.plot(t, ref_montage[name] + offset, color=colors[i], linewidth=1.2)
    ax.text(-0.05, offset, f'{name}-Avg', fontsize=10, fontweight='bold',
            color=colors[i], ha='right', va='center')
    offset -= 120

ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_title('Referential Montage (Average Reference)\nLook for the BIGGEST deflection',
             fontsize=14, fontweight='bold')
ax.set_yticks([])
plt.tight_layout()
plt.show()

print('In a referential montage, the electrode with the largest deflection')
print(f'is closest to the source. Here that is: {electrodes[source_idx]}')

**Exercise 3:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 4: Bipolar Montage (Adjacent Electrode Differences)

In a **bipolar montage**, each channel shows the difference between two neighboring electrodes along a chain.

$$\text{Channel} = V_{\text{electrode\_n}} - V_{\text{electrode\_{n+1}}}$$

The four channels in our chain are: Fp1-F3, F3-C3, C3-P3, P3-O1.

**The key insight:** When a channel deflects UP and the next channel deflects DOWN (or vice versa), the shared electrode between them is closest to the source. This is called a **phase reversal**.

In [ ]:
# Bipolar montage: differences between adjacent electrodes
bipolar_channels = [
    ('Fp1', 'F3'), ('F3', 'C3'), ('C3', 'P3'), ('P3', 'O1')
]
bipolar_labels = [f'{a}-{b}' for a, b in bipolar_channels]
bipolar_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

bipolar_signals = []
for a, b in bipolar_channels:
    bipolar_signals.append(voltages[a] - voltages[b])

# Plot bipolar montage
fig, ax = plt.subplots(figsize=(12, 7))
offset = 0
for i, (label, sig) in enumerate(zip(bipolar_labels, bipolar_signals)):
    ax.plot(t, sig + offset, color=bipolar_colors[i], linewidth=1.5)
    ax.text(-0.05, offset, label, fontsize=11, fontweight='bold',
            color=bipolar_colors[i], ha='right', va='center')
    offset -= 120

# Mark the phase reversal
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5)

# Arrow showing phase reversal
ax.annotate('PHASE\nREVERSAL', xy=(1.05, -120), fontsize=13,
            fontweight='bold', color='#c0392b',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#c0392b', alpha=0.9))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_title('Bipolar Montage (Left Parasagittal Chain)\n'
             'Look for where the deflection FLIPS direction = Phase Reversal',
             fontsize=14, fontweight='bold')
ax.set_yticks([])
plt.tight_layout()
plt.show()

print('F3-C3 deflects UP (because C3 is very negative, and we subtract it).')
print('C3-P3 deflects DOWN (because C3 is very negative, so the result is negative).')
print(f'The phase reversal between F3-C3 and C3-P3 localizes the source to: C3')

**Exercise 4:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 5: Side-by-Side: Referential vs Bipolar

Here both montages are displayed together for direct comparison. Same brain, same spike, different views:

- **Referential**: Look for the **biggest** deflection
- **Bipolar**: Look for where the deflection **flips direction** (phase reversal)

Both methods point to the same source electrode.

In [ ]:
# Side by side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Referential
offset = 0
for i, name in enumerate(electrodes):
    ax1.plot(t, ref_montage[name] + offset, color=colors[i], linewidth=1.2)
    ax1.text(-0.05, offset, f'{name}-Avg', fontsize=9, fontweight='bold',
            color=colors[i], ha='right', va='center')
    offset -= 100
ax1.axvline(1.0, color='gray', linestyle='--', alpha=0.4)
ax1.set_title('Referential Montage\n(find BIGGEST deflection)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Time (s)')
ax1.set_yticks([])

# Bipolar
offset = 0
for i, (label, sig) in enumerate(zip(bipolar_labels, bipolar_signals)):
    ax2.plot(t, sig + offset, color=bipolar_colors[i], linewidth=1.2)
    ax2.text(-0.05, offset, label, fontsize=9, fontweight='bold',
            color=bipolar_colors[i], ha='right', va='center')
    offset -= 100
ax2.axvline(1.0, color='gray', linestyle='--', alpha=0.4)
ax2.set_title('Bipolar Montage\n(find PHASE REVERSAL)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Time (s)')
ax2.set_yticks([])

plt.tight_layout()
plt.show()

print(f'Both montages localize the source to: {electrodes[source_idx]}')

**Exercise 5:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 6: Exercise 1: Move the Source to P3

**Your turn.** Move the spike source from C3 to **P3** (index 3) and regenerate both montages. Where does the phase reversal appear now?

**Expected:** In the bipolar montage, the phase reversal should shift to appear between C3-P3 and P3-O1 (the two channels that share P3).

**Exercise 6:** Try it yourself!

In [ ]:
# EXERCISE 1: Move the source to P3
# ===================================
# 1. Call generate_electrode_voltages with source_idx=3 (P3)
# 2. Compute the bipolar montage channels
# 3. Plot the bipolar montage
# 4. Where does the phase reversal appear now?

# YOUR CODE HERE
new_source_idx = 3  # P3
# new_voltages = generate_electrode_voltages(new_source_idx)
# ... compute bipolar channels ...
# ... plot ...

# Hint: The phase reversal should appear between C3-P3 and P3-O1,
# since both channels share P3, which is now the source.

## Step 7: Exercise 2: Equidistant Source

**Your turn.** What happens when the spike source is exactly **equidistant** between two electrodes? For example, halfway between F3 and C3.

Modify the simulation so both F3 and C3 receive equal-amplitude spikes. What does the bipolar montage look like now?

**Exercise 7:** Try it yourself!

In [ ]:
# EXERCISE 2: Source equidistant between two electrodes
# ====================================================
# When the source is exactly between F3 and C3, both electrodes
# see the same voltage spike. What happens to F3-C3?
#
# 1. Modify the voltage generation so F3 and C3 get the same spike amplitude
# 2. Compute and plot the bipolar montage
# 3. What happens to the F3-C3 channel? Why?

# YOUR CODE HERE

# Hint: If V(F3) = V(C3), then F3-C3 = 0.
# The spike DISAPPEARS in the channel where both electrodes are equidistant!
# This is called 'cancellation' -- a limitation of bipolar montages.
# The referential montage would still show the spike at both electrodes.

## Step 8: Summary

**Key takeaways:**

1. **A montage defines how signals are displayed**, not where electrodes go.
2. **Referential montage**: each electrode vs. a common reference. Look for the biggest deflection.
3. **Bipolar montage**: differences between adjacent electrodes. Look for the phase reversal.
4. **The double banana** (longitudinal bipolar) is the clinical workhorse -- four front-to-back chains.
5. **Cortical dipoles** are the source of EEG signals. Radial dipoles (on gyrus crowns) are well-detected; tangential dipoles (in sulci) may be invisible to surface EEG.
6. **Phase reversal** is the key localization tool -- the electrode where deflection flips direction is the source.

In [ ]:
print('Double Banana Montage -- 4 Chains')
print('=' * 55)
chains = [
    ('Left parasagittal',  'Fp1-F3, F3-C3, C3-P3, P3-O1'),
    ('Right parasagittal', 'Fp2-F4, F4-C4, C4-P4, P4-O2'),
    ('Left temporal',      'Fp1-F7, F7-T3, T3-T5, T5-O1'),
    ('Right temporal',     'Fp2-F8, F8-T4, T4-T6, T6-O2'),
]
for name, deriv in chains:
    print(f'{name:<22} {deriv}')

**Exercise 8:** Try it yourself!

In [ ]:
# YOUR CODE HERE
